<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/Sistem_automatizat_de_evaluare_a_%C3%AEmprumuturilor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn. impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_test_original = df_test.copy()

In [ ]:
df_train

,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status
0,CUST116708,40,Self-Employed,14.1,77335,679,4.4,67,17026,0,0,0,Personal Loan,Education,26200,12.04,0.220,0.339,0.113,1
1,CUST109237,18,Employed,0.0,34946,639,1.2,292,2620,0,1,0,Personal Loan,Education,37400,14.95,0.075,1.070,0.357,0
2,CUST102214,29,Employed,9.4,26778,595,5.5,191,4702,1,1,0,Personal Loan,Personal,52300,15.51,0.176,1.953,0.651,0
3,CUST114496,27,Student,1.5,25624,590,5.7,414,2609,0,0,0,Personal Loan,Debt Consolidation,4800,15.91,0.102,0.187,0.062,0
4,CUST127659,18,Employed,0.0,25676,568,1.8,19,3453,1,1,1,Personal Loan,Medical,24700,15.47,0.134,0.962,0.321,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,CUST100816,38,Employed,9.5,125144,613,13.7,428,16935,0,2,0,Line of Credit,Home Improvement,31800,11.40,0.135,0.254,0.085,1
39996,CUST106004,32,Self-Employed,4.3,161191,589,8.3,1773,71374,1,2,1,Credit Card,Medical,70000,22.84,0.443,0.434,0.145,0
39997,CUST141702,35,Self-Employed,5.1,115745,594,8.0,6965,31091,1,4,2,Personal Loan,Business,100000,15.52,0.269,0.864,0.288,0
39998,CUST144014,23,Self-Employed,1.0,43978,562,3.1,1147,22872,1,2,0,Personal Loan,Debt Consolidation,65300,16.79,0.520,1.485,0.495,0


In [ ]:
df_train.dtypes

,0
customer_id,object
age,int64
occupation_status,object
years_employed,float64
annual_income,int64
credit_score,int64
credit_history_years,float64
savings_assets,int64
current_debt,int64
defaults_on_file,int64


In [ ]:
def categorize_age(age):
    if age < 30:
        return 'Young'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'

def categorize_dti(dti):
    if dti < 20:
        return 'LowRisk'
    elif dti < 40:
        return 'MediumRisk'
    else:
        return 'HighRisk'

def impute_columns(df_train, df_test, columns, strategy):
    imputer = SimpleImputer(missing_values=np.nan, strategy=strategy)
    df_train[columns] = imputer.fit_transform(df_train[columns])
    df_test[columns] = imputer.transform(df_test[columns])

def encode_columns(df_train, df_test, columns):
    ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=int)
    encoded_train_list = ohe.fit_transform(df_train[columns])
    encoded_test_list = ohe.transform(df_test[columns])

    encoded_df_train = pd.DataFrame(encoded_train_list, columns=ohe.get_feature_names_out())
    encoded_df_test = pd.DataFrame(encoded_test_list, columns=ohe.get_feature_names_out())

    df_train[encoded_df_train.columns] = encoded_df_train
    df_test[encoded_df_test.columns] = encoded_df_test

    df_train.drop(columns, axis=1, inplace=True)
    df_test.drop(columns, axis=1, inplace=True)

def process(df_train, df_test):
    encode_list = ['occupation_status', 'product_type', 'loan_intent']
    encode_columns(df_train, df_test, encode_list)

In [ ]:
process(df_train, df_test)
df_train.head()

,customer_id,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,...,occupation_status_Student,product_type_Credit Card,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Business,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
0,CUST116708,40,14.1,77335,679,4.4,67,17026,0,0,...,0,0,0,1,0,0,1,0,0,0
1,CUST109237,18,0.0,34946,639,1.2,292,2620,0,1,...,0,0,0,1,0,0,1,0,0,0
2,CUST102214,29,9.4,26778,595,5.5,191,4702,1,1,...,0,0,0,1,0,0,0,0,0,1
3,CUST114496,27,1.5,25624,590,5.7,414,2609,0,0,...,1,0,0,1,0,1,0,0,0,0
4,CUST127659,18,0.0,25676,568,1.8,19,3453,1,1,...,0,0,0,1,0,0,0,0,1,0


In [ ]:
X = df_train.drop(['customer_id', 'loan_status'], axis=1)
y = df_train['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
def evaluate_model(model):
    y_train_pred = model.predict_proba(X_train)
    y_test_pred = model.predict_proba(X_test)

    y_train_pred = y_train_pred[:, 1]
    y_test_pred = y_test_pred[:, 1]

    print(f"Train: {roc_auc_score(y_train, y_train_pred)}, Test: {roc_auc_score(y_test, y_test_pred)}")

In [ ]:
rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, min_samples_split=3)
rf.fit(X_train, y_train)
evaluate_model(rf)

Train: 0.9991514221413509, Test: 0.9734104754923025


In [ ]:
lr = LogisticRegression()
lr.fit(X_train, y_train)
evaluate_model(lr)

Train: 0.9464387727967534, Test: 0.9453208498698147


In [ ]:
#depth = 6, iterations = 200, learning_rate = 0.1
ctb = CatBoostClassifier(verbose=False)
ctb.fit(X_train, y_train)
evaluate_model(ctb)

Train: 0.9945409087608413, Test: 0.9846208827321216


In [ ]:
xgb = XGBClassifier()
xgb.fit(X_train, y_train)
evaluate_model(xgb)

Train: 0.9980311438320867, Test: 0.9825300816501933


In [ ]:
lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)
evaluate_model(lgbm)

[LightGBM] [Info] Number of positive: 17598, number of negative: 14402
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006827 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2913
[LightGBM] [Info] Number of data points in the train set: 32000, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.549937 -> initscore=0.200418
[LightGBM] [Info] Start training from score 0.200418


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Train: 0.9911295325738229, Test: 0.9834182486918274


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
def solve(model):
    total_obligations = (
        df_test_original['current_debt'] +
        df_test_original['derogatory_marks'] +
        df_test_original['delinquencies_last_2yrs']
    ).astype(int)

    subtask1 = pd.DataFrame({
        'subtaskID': 1,
        'datapointID': df_test_original['customer_id'],
        'answer': df_test_original['age'].apply(categorize_age)
    })

    subtask2 = pd.DataFrame({
        'subtaskID': 2,
        'datapointID': df_test_original['customer_id'],
        'answer': df_test_original['debt_to_income_ratio'].apply(categorize_dti)
    })

    subtask3 = pd.DataFrame({
        'subtaskID': 3,
        'datapointID': df_test_original['customer_id'],
        'answer': total_obligations
    })

    X_pred_df = df_test.drop('customer_id', axis=1, errors='ignore')
    # Apply the same StandardScaler used for training data
    X_pred_scaled = sc.transform(X_pred_df)

    y_pred = model.predict_proba(X_pred_scaled)
    y_pred = y_pred[:, 1]

    subtask4 = pd.DataFrame({'subtaskID': 4, 'datapointID': df_test_original['customer_id'], 'answer': y_pred})
    output = pd.concat([subtask1, subtask2, subtask3, subtask4], axis=0)
    output.to_csv('submission.csv', index=False)

In [ ]:
solve(ctb)